# Compare CSVs

In [ ]:
# To compare by `vnnlib`

import pandas as pd
import numpy as np

A_PATH = r'../results/exp_2/triplets_k50176.0_eps0.0001_24_imgs.csv'
B_PATH = r'../results/exp_1/triplets_k50176.0_eps0.0001_239_imgs.csv'

dfA = pd.read_csv(A_PATH)
dfB = pd.read_csv(B_PATH)

# Basic sanity checks
required = {"vnnlib", "result"}
missingA = required - set(dfA.columns)
missingB = required - set(dfB.columns)
if missingA:
    raise ValueError(f"A is missing columns: {missingA}")
if missingB:
    raise ValueError(f"B is missing columns: {missingB}")

# Clean types
dfA["vnnlib"] = dfA["vnnlib"].astype(str)
dfB["vnnlib"] = dfB["vnnlib"].astype(str)

dfA["result"] = dfA["result"].astype(str)
dfB["result"] = dfB["result"].astype(str)

# (optional) normalize whitespace
dfA["result"] = dfA["result"].str.strip()
dfB["result"] = dfB["result"].str.strip()

In [17]:
# Unique vnnlibs
vA = set(dfA["vnnlib"].unique())
vB = set(dfB["vnnlib"].unique())

common = vA & vB
onlyA = vA - vB
onlyB = vB - vA

summary = pd.DataFrame({
    "metric": [
        "unique vnnlib in A",
        "unique vnnlib in B",
        "unique vnnlib in common",
        "unique vnnlib only in A",
        "unique vnnlib only in B",
        "rows in A",
        "rows in B",
    ],
    "value": [
        len(vA),
        len(vB),
        len(common),
        len(onlyA),
        len(onlyB),
        len(dfA),
        len(dfB),
    ]
})
summary

,metric,value
0,unique vnnlib in A,72
1,unique vnnlib in B,717
2,unique vnnlib in common,54
3,unique vnnlib only in A,18
4,unique vnnlib only in B,663
5,rows in A,72
6,rows in B,717


In [21]:
KEEP_COLS = [
    "vnnlib",
    "result",
    "all_time"
]

# helper to keep only columns that actually exist
def keep_existing(df, suffix):
    cols = [c for c in KEEP_COLS if c in df.columns]
    out = df[cols].copy()
    return out.rename(columns={c: f"{c}_{suffix}" for c in cols if c != "vnnlib"})

A_common = dfA[dfA["vnnlib"].isin(common)]
B_common = dfB[dfB["vnnlib"].isin(common)]

A_sel = keep_existing(A_common, "A")
B_sel = keep_existing(B_common, "B")

side_by_side = A_sel.merge(
    B_sel,
    on="vnnlib",
    how="outer"
)

print("Side-by-side shape:", side_by_side.shape)
side_by_side.head(-1)

Side-by-side shape: (54, 5)


,vnnlib,result_A,all_time_A,result_B,all_time_B
0,vnnlib/n01531178_goldfinch_global_k50176_eps_0...,sat True,9.812009,sat True,1864.432675
1,vnnlib/n01531178_goldfinch_seg0_fixmask_k50176...,sat True,141.021457,sat True,1953.363713
2,vnnlib/n01531178_goldfinch_seg0_fixnonmask_k50...,sat True,10.079423,sat True,2250.353688
3,vnnlib/n01580077_jay_global_k50176_eps_0.0001....,sat True,102.971542,sat True,1857.925682
4,vnnlib/n01580077_jay_seg0_fixmask_k50176_eps_0...,sat True,91.539672,sat True,1858.179778
5,vnnlib/n01580077_jay_seg0_fixnonmask_k50176_ep...,sat True,82.643588,sat True,1856.746706
6,vnnlib/n01631663_eft_global_k50176_eps_0.0001....,sat True,10.894804,sat True,2248.211634
7,vnnlib/n01631663_eft_seg0_fixmask_k50176_eps_0...,sat True,12.481471,sat True,1857.510090
8,vnnlib/n01631663_eft_seg0_fixnonmask_k50176_ep...,sat True,10.626640,sat True,1857.188028
9,vnnlib/n01641577_bullfrog_global_k50176_eps_0....,sat True,14.317746,sat True,1854.780201


----------------

In [7]:
# To see what images are common across all CSV files in a folder

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
CSV_DIR = Path("results/exp_1")     # <-- change if needed
IMAGE_COL = "image"        # column containing image names

# --------------------------------------------------
# LOAD ALL CSV FILES
# --------------------------------------------------
csv_files = sorted(CSV_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

if not csv_files:
    raise RuntimeError("No CSV files found!")

# --------------------------------------------------
# COLLECT IMAGE SETS
# --------------------------------------------------
image_sets = {}

for f in csv_files:
    df = pd.read_csv(f)

    if IMAGE_COL not in df.columns:
        raise ValueError(f"{f.name} does not contain column '{IMAGE_COL}'")

    imgs = set(df[IMAGE_COL].dropna().astype(str))
    image_sets[f.name] = imgs

    print(f"{f.name:45s} -> {len(imgs)} images")

# --------------------------------------------------
# INTERSECTION ACROSS ALL FILES
# --------------------------------------------------
common_images = set.intersection(*image_sets.values())

print("\n" + "=" * 70)
print(f"Images appearing in ALL {len(csv_files)} files: {len(common_images)}")
print("=" * 70)

# pretty display
common_images = sorted(common_images)

for img in common_images:
    print(img)

Found 4 CSV files
triplets_k10.0_eps0.0001_278_imgs.csv         -> 278 images
triplets_k10.0_eps0.0003_277_imgs.csv         -> 277 images
triplets_k50176.0_eps0.0001_239_imgs.csv      -> 239 images
triplets_k50176.0_eps0.0003_160_imgs.csv      -> 160 images

Images appearing in ALL 4 files: 156
n01440764_tench
n01514668_cock
n01514859_hen
n01518878_ostrich
n01531178_goldfinch
n01580077_jay
n01631663_eft
n01641577_bullfrog
n01685808_whiptail
n01687978_agama
n01704323_triceratops
n01756291_sidewinder
n01768244_trilobite
n01770393_scorpion
n01774750_tarantula
n01796340_ptarmigan
n01807496_partridge
n01818515_macaw
n01820546_lorikeet
n01824575_coucal
n01829413_hornbill
n01843065_jacamar
n01847000_drake
n01871265_tusker
n01872401_echidna
n01873310_platypus
n01877812_wallaby
n01882714_koala
n01883070_wombat
n01945685_slug
n01990800_isopod
n02007558_flamingo
n02051845_pelican
n02058221_albatross
n02074367_dugong
n02086240_Shih-Tzu
n02086910_papillon
n02091831_Saluki
n02096051_Airedale
n021015

In [8]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
INPUT_DIR = Path("results/exp_1_only_global")                 # folder with original CSVs
OUTPUT_DIR = Path("results/exp_1_only_global_common")        # NEW folder for filtered CSVs

IMAGE_COL = "image"

OUTPUT_DIR.mkdir(exist_ok=True)

# --------------------------------------------------
# LOAD FILES
# --------------------------------------------------
csv_files = sorted(INPUT_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

if not csv_files:
    raise RuntimeError("No CSV files found!")

# --------------------------------------------------
# COLLECT IMAGE SETS
# --------------------------------------------------
image_sets = {}

for f in csv_files:
    df = pd.read_csv(f)

    if IMAGE_COL not in df.columns:
        raise ValueError(f"{f.name} missing '{IMAGE_COL}'")

    image_sets[f.name] = set(df[IMAGE_COL].dropna().astype(str))

# --------------------------------------------------
# INTERSECTION
# --------------------------------------------------
common_images = set.intersection(*image_sets.values())

print(f"\nImages common to ALL files: {len(common_images)}")

# --------------------------------------------------
# FILTER + SAVE NEW CSVs
# --------------------------------------------------
for f in csv_files:
    df = pd.read_csv(f)

    df_common = df[df[IMAGE_COL].astype(str).isin(common_images)].copy()

    out_path = OUTPUT_DIR / f.name
    df_common.to_csv(out_path, index=False)

    print(f"{f.name:45s} -> saved {len(df_common)} rows")

print("\nDONE")
print(f"Filtered CSVs written to: {OUTPUT_DIR.resolve()}")

Found 2 CSV files

Images common to ALL files: 157
triplets_k50176.0_eps0.0001_239_imgs.csv      -> saved 471 rows
triplets_k50176.0_eps0.0003_160_imgs.csv      -> saved 471 rows

DONE
Filtered CSVs written to: /Users/zd3504phd/Desktop/XAIV/analysis/results/exp_1_only_global_common
